# TUGAS MANDIRI — Pertemuan 7
## Analisis Performa Kedai Kopi Nusantara: Pipeline End-to-End

| | |
|---|---|
| **Nama** | Hilmi Mufid |
| **NIM** | 2505060046 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 7 |

Tenggat: dikumpulkan paling lambat sebelum UTS dimulai. Sifat tugas: individu.

Konteks: saya direkrut sebagai data analyst paruh waktu oleh "Kopi Nusantara", jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus terpandu pada modul (Sub-bab 7.1-7.4), namun dengan data dan struktur yang sedikit berbeda.

Pipeline dibangun mengikuti seluruh tahap Extract, Transform (join, window function, Spark SQL), Load, ditutup dengan ringkasan eksekutif dan rekomendasi bisnis.

Catatan lingkungan dan path: dijalankan di laptop Ubuntu (dual boot), Spark/PySpark 3.5.9, Hadoop 3.4.3. Mengikuti konvensi direktori kerja pribadi yang dipakai sejak Pertemuan 3 (`/home/mufid/praktikum-bigdata0046`), maka `/user/[username]/tugas7` pada instruksi soal disesuaikan menjadi `/home/mufid/praktikum-bigdata0046/tugas7`.

Persiapan: membuat tiga sumber data dan mengunggah data transaksi ke HDFS

Tiga sumber data disiapkan sesuai instruksi modul: data cabang (JSON, 3 baris), data menu (CSV, 15 baris), dan data transaksi (CSV, 4000 baris, data utama).

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /home/mufid/praktikum-bigdata0046/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /home/mufid/praktikum-bigdata0046/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /home/mufid/praktikum-bigdata0046/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /home/mufid/praktikum-bigdata0046/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-PipelineKopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


A. EXTRACT (bobot 10%)

Ketiga sumber data dibaca dari HDFS (bukan disk lokal) menjadi tiga Spark DataFrame, sesuai format aslinya masing-masing.

In [5]:
BASE_PATH = "hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas7/raw"

df_transaksi = spark.read.csv(f"{BASE_PATH}/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json(f"{BASE_PATH}/tugas7_cabang.json")
df_menu = spark.read.csv(f"{BASE_PATH}/tugas7_menu.csv", header=True, inferSchema=True)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())
df_transaksi.printSchema()
df_cabang.printSchema()
df_menu.printSchema()

Transaksi: 4000 | Cabang: 3 | Menu: 15
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)

root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)

root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)



B. TRANSFORM — Join dan Kolom Turunan (bobot 20%)

Ketiga tabel digabungkan dengan inner join (cabang_id dan menu_id sama-sama data referensi yang lengkap, sehingga aman memakai inner join tanpa risiko kehilangan baris), lalu ditambahkan kolom total_penjualan (qty x harga).

In [6]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

print("Jumlah baris setelah join:", df_gabungan.count())
df_gabungan.show(5)

Jumlah baris setelah join: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makana

C. TRANSFORM — Window Function (bobot 20%)

Mengikuti pola persis Sub-bab 7.3.2 pada modul: total penjualan diringkas dulu per kombinasi cabang + menu, lalu window dipartisi per nama_cabang dan diambil top-2 memakai row_number().

In [7]:
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                  .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(10)

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



D. TRANSFORM — Spark SQL (bobot 20%)

Menggunakan Spark SQL murni lewat spark.sql(), memanfaatkan temporary view yang didaftarkan dari df_gabungan.

In [8]:
df_gabungan.createOrReplaceTempView("gabungan_tugas7")

total_per_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan_tugas7
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

total_per_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



E. LOAD (bobot 15%)

Data gabungan (hasil bagian B) disimpan ke HDFS sebagai Parquet, dipartisi berdasarkan kategori_menu, lalu diverifikasi dengan membaca kembali dan menghitung jumlah baris.

In [9]:
!hdfs dfs -mkdir -p /home/mufid/praktikum-bigdata0046/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas7/processed/laporan"
)

print("Data gabungan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.")
!hdfs dfs -ls /home/mufid/praktikum-bigdata0046/tugas7/processed/laporan

[Stage 37:>                                                         (0 + 1) / 1]

Data gabungan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.


Found 4 items
-rw-r--r--   3 mufid supergroup          0 2026-10-02 07:12 /home/mufid/praktikum-bigdata0046/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - mufid supergroup          0 2026-10-02 07:12 /home/mufid/praktikum-bigdata0046/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - mufid supergroup          0 2026-10-02 07:12 /home/mufid/praktikum-bigdata0046/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - mufid supergroup          0 2026-10-02 07:12 /home/mufid/praktikum-bigdata0046/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [10]:
# Verifikasi: membaca kembali dari HDFS dan menghitung jumlah baris
df_verifikasi = spark.read.parquet("hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas7/processed/laporan")
print("Jumlah baris hasil akhir (dibaca kembali dari HDFS):", df_verifikasi.count())

Jumlah baris hasil akhir (dibaca kembali dari HDFS): 4000


F. Ringkasan Eksekutif dan Rekomendasi (bobot 15%)

Tabel ringkasan per cabang (total penjualan, jumlah transaksi, rata-rata nilai transaksi) disusun dari data yang sudah diverifikasi di bagian E.

In [11]:
ringkasan_eksekutif_tugas7 = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif_tugas7.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



Hasil ringkasan eksekutif:

| nama_cabang | kota | kepala_barista | total_penjualan | jumlah_transaksi | rata_rata_nilai_transaksi |
|---|---|---|---|---|---|
| Kopi Nusantara Semarang | Semarang | Nadia | 90.542.000 | 1.393 | 64.998 |
| Kopi Nusantara Magelang | Magelang | Nadia | 87.138.000 | 1.302 | 66.926 |
| Kopi Nusantara Yogyakarta | Yogyakarta | Reza | 84.645.000 | 1.305 | 64.862 |

Rekomendasi dan interpretasi bisnis:

Berdasarkan hasil ringkasan eksekutif, cabang Kopi Nusantara Semarang di bawah kepala barista Nadia adalah cabang dengan kinerja terbaik, mencatat total penjualan Rp90.542.000 dari 1.393 transaksi selama periode ini — unggul tipis dari Kopi Nusantara Magelang (Rp87.138.000, juga di bawah Nadia) dan Kopi Nusantara Yogyakarta (Rp84.645.000, di bawah Reza). Meski begitu, selisih ketiganya relatif tipis (kurang dari 7% antara yang tertinggi dan terendah), menunjukkan performa ketiga cabang sebenarnya cukup merata, dan cabang Yogyakarta berada di posisi kinerja terlemah dengan rata-rata nilai transaksi terendah (Rp64.862).

Menariknya, kepala barista Nadia bertanggung jawab atas dua cabang (Semarang dan Magelang) yang menempati dua posisi teratas, sementara Reza yang menangani Yogyakarta seorang diri berada di posisi terakhir — pola ini pantas ditelusuri lebih lanjut oleh manajemen, apakah karena kepemimpinan Nadia yang lebih efektif, atau karena faktor lokasi dan demografi pelanggan yang berbeda di tiap kota.

Untuk menu yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (Yogyakarta), berdasarkan temuan bagian C, Menu-12 (kategori Makanan Ringan) tercatat sebagai menu dengan penjualan tertinggi di Kopi Nusantara Yogyakarta (Rp9.275.000), diikuti Menu-15 (kategori Non-Kopi, Rp8.160.000). Keduanya sudah menjadi andalan cabang ini, sehingga strategi paling masuk akal adalah memperkuat promosi bundling kedua menu tersebut (misalnya paket kopi + makanan ringan dengan harga spesial) untuk mendorong nilai transaksi rata-rata naik, alih-alih memperkenalkan menu baru yang belum terbukti diminati pelanggan di cabang tersebut.

Menutup SparkSession

In [12]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
